### 1. Instalación y configuración inicial

In [ ]:
import sys #accede a la información del sistema, como la versión de Python y los módulos instalados
#instala la biblioteca findspark, que es para encontrar la ubicación de Spark en el sistema y configurarla para su uso en Python.
!"{sys.executable}" -m pip install findspark 

In [ ]:
import findspark
# Inicializa findspark con la ruta donde se encuentra Spark en el sistema. 
# Es necesario para que Python pueda encontrar y usar Spark correctamente.
findspark.init(r"C:\Users\Alejandro Mondragon\Documents\spark\spark-4.1.1-bin-hadoop3")

In [ ]:

#Verifica que Spark fue encontrado correctamente.
findspark.find()

'C:\\Users\\Alejandro Mondragon\\Documents\\spark\\spark-4.1.1-bin-hadoop3'

### 2. Creación de la sesión spark

In [ ]:
# Clases base para configurar Spark.
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
# Crea una sesión Spark que usa todos los núcleos disponibles de la computadora.
spark = SparkSession.builder.master("local[*]").getOrCreate()
# Configura parámetros de Spark
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
#Muestra información de la sesión (versión, master, appName, etc.).
spark

### 3. Lectura y limpieza de datos

In [6]:
spark = SparkSession.builder.appName("NYC Yellow Cab").getOrCreate()

In [ ]:
from pyspark.sql import functions as F

def read_and_clean(path, rename_ratecode=False):
    df = spark.read.csv(  #Carga un archivo CSV como DataFrame.
        path,
        header=True,
        sep=",",
        inferSchema=False  #No infiere tipos automáticamente (se cargan como strings).
    )

    # Renombra la columna RateCodeID si es necesario (para uniformidad entre datasets).
    if rename_ratecode:
        df = df.withColumnRenamed("RateCodeID", "RatecodeID")

    # Convierte columnas a tipos correctos (int, double, timestamp, string).
    return (
        df
        .withColumn("VendorID", F.col("VendorID").cast("int"))
        .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
        .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("trip_distance", F.col("trip_distance").cast("double"))
        .withColumn("pickup_longitude", F.col("pickup_longitude").cast("double"))
        .withColumn("pickup_latitude", F.col("pickup_latitude").cast("double"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
        .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast("string"))
        .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double"))
        .withColumn("dropoff_latitude", F.col("dropoff_latitude").cast("double"))
        .withColumn("payment_type", F.col("payment_type").cast("int"))
        .withColumn("fare_amount", F.col("fare_amount").cast("double"))
        .withColumn("extra", F.col("extra").cast("double"))
        .withColumn("mta_tax", F.col("mta_tax").cast("double"))
        .withColumn("tip_amount", F.col("tip_amount").cast("double"))
        .withColumn("tolls_amount", F.col("tolls_amount").cast("double"))
        .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast("double"))
        .withColumn("total_amount", F.col("total_amount").cast("double"))
    )

### 4. Unión de datasets

In [ ]:
# Aplica la función de limpieza a varios archivos.
df1 = read_and_clean(
    "C:/Users/Alejandro Mondragon/Documents/Tec/Big Data/Semana 2/yellow_tripdata_2015-01.csv",
    rename_ratecode=True
)

df2 = read_and_clean("C:/Users/Alejandro Mondragon/Documents/Tec/Big Data/Semana 2/yellow_tripdata_2016-01.csv")
df3 = read_and_clean("C:/Users/Alejandro Mondragon/Documents/Tec/Big Data/Semana 2/yellow_tripdata_2016-02.csv")
df4 = read_and_clean("C:/Users/Alejandro Mondragon/Documents/Tec/Big Data/Semana 2/yellow_tripdata_2016-03.csv")

# Une los DataFrames por nombre de columna
df = df1.unionByName(df2).unionByName(df3).unionByName(df4)

In [8]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|  -73.993896484375|  40.7501106262207|         1|    

### 5. Estadísticas descriptivas

In [ ]:
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount"
]

# Calcula estadísticas básicas para columnas numéricas.
df.select(numeric_cols).describe().show()

+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|   passenger_count|    trip_distance|       fare_amount|       tip_amount|      tolls_amount|      total_amount|
+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|          47248845|         47248845|          47248845|         47248845|          47248845|          47248845|
|   mean|1.6670397339871483|7.508417945877774|12.392189114675677|1.794568086055874|0.2843674354367921|15.592732594300593|
| stddev|1.3220922307333782|6487.658339592023| 78.61770040483957|574.7383884994606| 1.657184394696862| 580.1392733880709|
|    min|                 0|       -3390583.8|            -957.6|           -220.8|            -99.99|            -958.4|
|    max|                 9|     1.90726288E7|         429496.72|        3950588.8|           1450.09|         3950611.6|
+-------+---------------

### 6. Valores mínimos y máximos

In [ ]:
from pyspark.sql import functions as F

# Obtiene los valores mínimo y máximo de distancia, tarifa y total.
df.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),

    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),

    F.min("total_amount").alias("min_total"),
    F.max("total_amount").alias("max_total")
).show()

+------------+------------+--------+---------+---------+---------+
|min_distance|max_distance|min_fare| max_fare|min_total|max_total|
+------------+------------+--------+---------+---------+---------+
|  -3390583.8|1.90726288E7|  -957.6|429496.72|   -958.4|3950611.6|
+------------+------------+--------+---------+---------+---------+



### 7. Agrupaciones y conteo

In [ ]:
# Agrupa por tipo de pago y cuenta cuántos viajes hubo con cada método
df.groupBy("payment_type") \
  .count() \
  .orderBy(F.desc("count")) \
  .show()

+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|30870614|
|           2|16158086|
|           3|  164138|
|           4|   56004|
|           5|       3|
+------------+--------+



In [ ]:
# Agrupa por número de pasajeros
df.groupBy("passenger_count") \
  .count() \
  .orderBy("passenger_count") \
  .show()

+---------------+--------+
|passenger_count|   count|
+---------------+--------+
|              0|    8214|
|              1|33537914|
|              2| 6719430|
|              3| 1912291|
|              4|  911351|
|              5| 2551660|
|              6| 1607758|
|              7|      81|
|              8|      78|
|              9|      68|
+---------------+--------+



In [ ]:
# Agrupa por empresa proveedora del taxi
df.groupBy("VendorID").count().show()

+--------+--------+
|VendorID|   count|
+--------+--------+
|       1|22227251|
|       2|25021594|
+--------+--------+

